In [1]:
pip install selenium beautifulsoup4 pandas requests


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 21.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.7/153.7 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 12.5 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
  Attempting uninstall: certifi
    Found existing installation: certifi 2025.11.12
    Uninstalling certifi-2025.11.12:
      Successfully uninstalled certifi-2025.11.12
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
aext-assistant-server 0.4.0 requires anaconda-cloud-auth, which is not installed.
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install selenium 

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install beautifulsoup4

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install requests

Note: you may need to restart the kernel to use updated packages.


In [1]:
"""
Las Vegas Strip — Review Scraping Pipeline
===========================================
Scrapes TripAdvisor review data for every entity listed in
Data/las_vegas_strip_entities.md and writes results to
Data/las_vegas_reviews.csv.

Libraries: selenium, beautifulsoup4, pandas, requests
Run:  python scrape_reviews.py
"""

import os
import re
import csv
import time
import random
import logging
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    TimeoutException,
    NoSuchElementException,
    WebDriverException,
    StaleElementReferenceException,
)

# ──────────────────────────────────────────────
# Configuration
# ──────────────────────────────────────────────
BASE_DIR = Path(__file__).resolve().parent
DATA_DIR = BASE_DIR / "Data"
INPUT_FILE = DATA_DIR / "las_vegas_strip_entities.md"
OUTPUT_FILE = DATA_DIR / "las_vegas_reviews.csv"

TRIPADVISOR_BASE = "https://www.tripadvisor.com"
SEARCH_URL = f"{TRIPADVISOR_BASE}/Search"

# Date window for reviews (inclusive)
DATE_START = datetime(2023, 1, 1)
DATE_END = datetime(2026, 12, 31)

# Throttle settings (seconds)
PAGE_DELAY_MIN = 3
PAGE_DELAY_MAX = 7
REQUEST_TIMEOUT = 30

# Retry settings
MAX_RETRIES = 3
RETRY_BACKOFF = 5  # seconds, multiplied by attempt number

# Maximum review pages to scrape per entity (safety cap)
MAX_PAGES_PER_ENTITY = 50

# Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  [%(levelname)s]  %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)


# ──────────────────────────────────────────────
# 1. Parse Entity List from Markdown
# ──────────────────────────────────────────────
def parse_entity_list(filepath: Path) -> list[str]:
    """
    Read the markdown table and return a deduplicated list of entity names.
    Skips header / separator rows.
    """
    entities = []
    with open(filepath, "r", encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line.startswith("|"):
                continue
            cols = [c.strip() for c in line.split("|")]
            # cols[0] and cols[-1] are empty strings from leading/trailing pipes
            if len(cols) < 3:
                continue
            name = cols[1]
            # Skip header and separator rows
            if name in ("", "Entity Name") or set(name) <= {"-", " "}:
                continue
            entities.append(name)
    # Deduplicate while preserving order
    seen = set()
    unique = []
    for e in entities:
        if e not in seen:
            seen.add(e)
            unique.append(e)
    logger.info(f"Loaded {len(unique)} entities from {filepath.name}")
    return unique


# ──────────────────────────────────────────────
# 2. Initialize Selenium WebDriver
# ──────────────────────────────────────────────
def init_driver() -> webdriver.Chrome:
    """
    Create a headless Chrome WebDriver with anti-detection tweaks.
    """
    chrome_opts = Options()
    chrome_opts.add_argument("--headless=new")
    chrome_opts.add_argument("--disable-gpu")
    chrome_opts.add_argument("--no-sandbox")
    chrome_opts.add_argument("--disable-dev-shm-usage")
    chrome_opts.add_argument("--window-size=1920,1080")
    chrome_opts.add_argument(
        "user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    )
    # Reduce Selenium detection fingerprint
    chrome_opts.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_opts.add_experimental_option("useAutomationExtension", False)

    driver = webdriver.Chrome(options=chrome_opts)
    # Override navigator.webdriver flag
    driver.execute_cdp_cmd(
        "Page.addScriptToEvaluateOnNewDocument",
        {"source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"},
    )
    logger.info("Chrome WebDriver initialised (headless)")
    return driver


# ──────────────────────────────────────────────
# 3. Search TripAdvisor for Entity Page URL
# ──────────────────────────────────────────────
def search_entity(driver: webdriver.Chrome, entity_name: str) -> str | None:
    """
    Use TripAdvisor search to find the main page URL for the given entity.
    Returns the URL string or None if not found.
    """
    query = f"{entity_name} Las Vegas"
    search_url = f"{SEARCH_URL}?q={requests.utils.quote(query)}"

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            logger.info(f"  Searching TripAdvisor for '{entity_name}' (attempt {attempt})")
            driver.get(search_url)
            time.sleep(random.uniform(3, 5))

            # Wait for search results container
            WebDriverWait(driver, REQUEST_TIMEOUT).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "div[data-test-attribute='search-results'],"
                                                                  " div.search-results-list,"
                                                                  " div.prw_search_search_result_poi"))
            )

            soup = BeautifulSoup(driver.page_source, "html.parser")

            # Look for result links pointing to Hotel_Review, Restaurant_Review, or Attraction_Review
            link_pattern = re.compile(r"/(Hotel_Review|Restaurant_Review|Attraction_Review)-")
            for a_tag in soup.find_all("a", href=link_pattern):
                href = a_tag.get("href", "")
                if href.startswith("/"):
                    href = TRIPADVISOR_BASE + href
                # Basic relevance check — entity words should appear in the link text or URL
                link_text = a_tag.get_text(strip=True).lower()
                entity_lower = entity_name.lower()
                first_word = entity_lower.split()[0]
                if first_word in link_text or first_word in href.lower():
                    logger.info(f"  → Found URL: {href[:120]}...")
                    return href

            logger.warning(f"  No matching result for '{entity_name}' on attempt {attempt}")
            return None

        except (TimeoutException, WebDriverException) as exc:
            logger.warning(f"  Search error (attempt {attempt}): {exc}")
            time.sleep(RETRY_BACKOFF * attempt)

    return None


# ──────────────────────────────────────────────
# 4. Parse Review Date Strings
# ──────────────────────────────────────────────
# Common TripAdvisor date patterns
_MONTH_MAP = {
    "jan": 1, "feb": 2, "mar": 3, "apr": 4, "may": 5, "jun": 6,
    "jul": 7, "aug": 8, "sep": 9, "oct": 10, "nov": 11, "dec": 12,
}

def parse_review_date(raw: str) -> datetime | None:
    """
    Parse various TripAdvisor date formats:
      • 'March 2024'
      • 'Mar 2024'
      • 'Date of visit: February 2025'
      • '3/15/2024'
      • 'Written January 10, 2024'
    Returns a datetime or None if parsing fails.
    """
    if not raw:
        return None
    raw = raw.strip()
    # Remove common prefixes
    for prefix in ("Date of visit:", "Written", "Reviewed"):
        raw = raw.replace(prefix, "").strip()
    raw = raw.strip(": ")

    # Try full date with day, e.g. "January 10, 2024"
    for fmt in ("%B %d, %Y", "%b %d, %Y", "%m/%d/%Y", "%Y-%m-%d"):
        try:
            return datetime.strptime(raw, fmt)
        except ValueError:
            continue

    # Try month-year only, e.g. "March 2024" or "Mar 2024"
    for fmt in ("%B %Y", "%b %Y"):
        try:
            return datetime.strptime(raw, fmt)
        except ValueError:
            continue

    return None


# ──────────────────────────────────────────────
# 5. Extract Rating from a Review Element
# ──────────────────────────────────────────────
def extract_rating(review_soup) -> float | None:
    """
    Look for the bubble rating element and return a numeric score (1-5).
    TripAdvisor encodes the rating in a class like 'bubble_50' (= 5.0).
    """
    # Pattern 1: class containing 'bubble_XX'
    bubble = review_soup.find(class_=re.compile(r"bubble_\d+"))
    if bubble:
        cls_match = re.search(r"bubble_(\d+)", " ".join(bubble.get("class", [])))
        if cls_match:
            return int(cls_match.group(1)) / 10.0

    # Pattern 2: SVG title element (newer layout)
    svg_title = review_soup.find("title", string=re.compile(r"\d.*of.*5"))
    if svg_title:
        num = re.search(r"([\d.]+)\s*of\s*5", svg_title.text)
        if num:
            return float(num.group(1))

    # Pattern 3: aria-label on a span/div
    rated = review_soup.find(attrs={"aria-label": re.compile(r"\d.*of.*5")})
    if rated:
        num = re.search(r"([\d.]+)\s*of\s*5", rated["aria-label"])
        if num:
            return float(num.group(1))

    return None


# ──────────────────────────────────────────────
# 6. Scrape Reviews from a Single Entity Page
# ──────────────────────────────────────────────
def scrape_entity_reviews(
    driver: webdriver.Chrome,
    entity_name: str,
    entity_url: str,
) -> list[dict]:
    """
    Navigate through review pages for one entity and collect reviews
    within the target date range.
    Returns a list of dicts with keys:
        location_name, review_date, rating, review_text
    """
    reviews: list[dict] = []
    page_num = 0

    current_url = entity_url

    while page_num < MAX_PAGES_PER_ENTITY:
        page_num += 1
        logger.info(f"  Page {page_num} — {current_url[:100]}...")

        for attempt in range(1, MAX_RETRIES + 1):
            try:
                driver.get(current_url)
                time.sleep(random.uniform(PAGE_DELAY_MIN, PAGE_DELAY_MAX))

                # Expand truncated review text if possible
                try:
                    more_buttons = driver.find_elements(By.CSS_SELECTOR,
                        "span.Ignyf, span[onclick*='ReadMore'], div.prw_reviews_text_summary_hsx span[onclick]")
                    for btn in more_buttons:
                        try:
                            driver.execute_script("arguments[0].click();", btn)
                            time.sleep(0.3)
                        except StaleElementReferenceException:
                            pass
                except Exception:
                    pass  # Continue even if expansion fails

                break  # successful page load
            except (TimeoutException, WebDriverException) as exc:
                logger.warning(f"  Page load error (attempt {attempt}): {exc}")
                time.sleep(RETRY_BACKOFF * attempt)
        else:
            logger.error(f"  Failed to load page after {MAX_RETRIES} retries — stopping entity")
            break

        # Parse page HTML
        soup = BeautifulSoup(driver.page_source, "html.parser")

        # Identify individual review containers
        review_cards = soup.find_all("div", attrs={
            "data-test-target": re.compile(r"review", re.I)
        })
        if not review_cards:
            # Fallback: look for common review container classes
            review_cards = soup.find_all("div", class_=re.compile(
                r"review-container|reviewSelector|_c$|YibKl"
            ))

        if not review_cards:
            logger.info(f"  No review cards found on page {page_num} — stopping pagination")
            break

        out_of_range_count = 0

        for card in review_cards:
            # --- Rating ---
            rating = extract_rating(card)
            if rating is None:
                continue  # Skip reviews missing ratings per requirements

            # --- Date ---
            date_el = card.find(class_=re.compile(r"date|ratingDate|cRVS"))
            if not date_el:
                date_el = card.find("span", string=re.compile(r"(20\d{2})", re.I))
            raw_date = date_el.get_text(strip=True) if date_el else ""
            # Also check title attribute (some layouts)
            if not raw_date and date_el:
                raw_date = date_el.get("title", "")

            review_date = parse_review_date(raw_date)

            if review_date is None:
                # Handle missing / malformed date: skip but log
                logger.debug(f"    Skipping review with unparseable date: '{raw_date}'")
                continue

            # Check date window
            if review_date < DATE_START:
                out_of_range_count += 1
                continue
            if review_date > DATE_END:
                continue

            # --- Review Text ---
            text_el = card.find("q") or card.find(class_=re.compile(
                r"partial_entry|review-text|entry|QewHA"
            ))
            review_text = text_el.get_text(strip=True) if text_el else ""

            reviews.append({
                "location_name": entity_name,
                "review_date": review_date.strftime("%Y-%m-%d"),
                "rating": rating,
                "review_text": review_text,
            })

        logger.info(f"  Collected {len(reviews)} reviews so far for '{entity_name}'")

        # If most reviews on this page are too old, stop pagination
        if out_of_range_count > len(review_cards) * 0.5 and page_num > 1:
            logger.info("  Majority of reviews on page are before 2023 — stopping pagination")
            break

        # --- Find next-page link ---
        next_link = None
        next_btn = soup.find("a", class_=re.compile(r"next|nav.next", re.I))
        if next_btn and next_btn.get("href"):
            next_link = next_btn["href"]
        else:
            # Fallback: look for pagination links with offset pattern
            pagination_links = soup.find_all("a", href=re.compile(r"or\d+", re.I))
            if pagination_links:
                # Pick the link right after the current offset
                next_link = pagination_links[-1].get("href")

        if not next_link:
            logger.info("  No next-page link found — pagination complete")
            break

        if next_link.startswith("/"):
            next_link = TRIPADVISOR_BASE + next_link
        current_url = next_link

    return reviews


# ──────────────────────────────────────────────
# 7. Write Reviews to CSV (Incremental)
# ──────────────────────────────────────────────
def init_csv(filepath: Path) -> None:
    """Create the CSV file with header row."""
    with open(filepath, "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=["location_name", "review_date", "rating", "review_text"])
        writer.writeheader()
    logger.info(f"Initialised output CSV: {filepath}")


def append_reviews_to_csv(filepath: Path, reviews: list[dict]) -> None:
    """Append a batch of review dicts to the CSV file."""
    if not reviews:
        return
    with open(filepath, "a", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=["location_name", "review_date", "rating", "review_text"])
        writer.writerows(reviews)
    logger.info(f"  Wrote {len(reviews)} reviews to {filepath.name}")


# ──────────────────────────────────────────────
# 8. Main Orchestrator
# ──────────────────────────────────────────────
def main():
    logger.info("=" * 60)
    logger.info("Las Vegas Strip — Review Scraping Pipeline")
    logger.info("=" * 60)

    # Step 1: Load entity list
    entities = parse_entity_list(INPUT_FILE)
    if not entities:
        logger.error("No entities found — exiting")
        return

    # Step 2: Prepare output CSV
    init_csv(OUTPUT_FILE)

    # Step 3: Launch browser
    driver = init_driver()

    total_reviews = 0

    try:
        for idx, entity_name in enumerate(entities, start=1):
            logger.info(f"\n[{idx}/{len(entities)}] Processing: {entity_name}")

            # Search for the entity's TripAdvisor page
            entity_url = search_entity(driver, entity_name)
            if not entity_url:
                logger.warning(f"  Could not find TripAdvisor page — skipping '{entity_name}'")
                continue

            # Scrape reviews
            reviews = scrape_entity_reviews(driver, entity_name, entity_url)

            # Save incrementally
            append_reviews_to_csv(OUTPUT_FILE, reviews)
            total_reviews += len(reviews)

            logger.info(f"  ✓ {entity_name}: {len(reviews)} reviews | Running total: {total_reviews}")

            # Throttle between entities
            sleep_sec = random.uniform(PAGE_DELAY_MIN, PAGE_DELAY_MAX)
            logger.info(f"  Sleeping {sleep_sec:.1f}s before next entity...")
            time.sleep(sleep_sec)

    except KeyboardInterrupt:
        logger.info("\nInterrupted by user — saving progress and exiting")

    finally:
        driver.quit()
        logger.info("WebDriver closed")

    # Summary
    logger.info("=" * 60)
    logger.info(f"Pipeline complete. Total reviews collected: {total_reviews}")
    logger.info(f"Output saved to: {OUTPUT_FILE}")
    logger.info("=" * 60)

    # Quick preview
    if OUTPUT_FILE.exists():
        df = pd.read_csv(OUTPUT_FILE)
        logger.info(f"\nDataset shape: {df.shape}")
        logger.info(f"Date range: {df['review_date'].min()} → {df['review_date'].max()}")
        logger.info(f"Locations represented: {df['location_name'].nunique()}")
        logger.info(f"\nRating distribution:\n{df['rating'].value_counts().sort_index()}")


if __name__ == "__main__":
    main()

NameError: name '__file__' is not defined